# LIMPIEZA DE DATOS
## PASO 1: Importacion y carga inicial


In [1]:
import pandas as pd
df_covid = pd.read_csv("positivos_covid 3.csv", sep=";")
df_covid.head()

,FECHA_CORTE,DEPARTAMENTO,PROVINCIA,DISTRITO,METODODX,EDAD,SEXO,FECHA_RESULTADO,UBIGEO,id_persona
0,20241203,TUMBES,TUMBES,TUMBES,AG,46.0,FEMENINO,20221207.0,240101.0,203499.0
1,20241203,LIMA,LIMA,JESUS MARIA,AG,69.0,FEMENINO,20230822.0,150113.0,221397.0
2,20241203,SAN MARTIN,MOYOBAMBA,MOYOBAMBA,AG,55.0,FEMENINO,20240108.0,220101.0,295651.0
3,20241203,AREQUIPA,CAYLLOMA,COPORAQUE,AG,50.0,MASCULINO,20230824.0,40506.0,851625.0
4,20241203,LIMA,LIMA,JESUS MARIA,AG,58.0,MASCULINO,20221217.0,150113.0,287786.0


## PASO 2:  Análisis de valores nulos
### Objetivo: Identificar columnas con valores faltantes

In [2]:
# Conteo de nulos
df_covid.isnull().sum()
# Porcentaje de nulos
(df_covid.isnull().sum() / len(df_covid) * 100).round(2)

FECHA_CORTE         0.00
DEPARTAMENTO        0.00
PROVINCIA           0.00
DISTRITO            0.00
METODODX            0.00
EDAD                0.01
SEXO                0.00
FECHA_RESULTADO     0.04
UBIGEO              5.16
id_persona         99.73
dtype: float64

## PASO 3: Análisis de duplicados
### Objetivo: Detectar registros duplicados

In [3]:
# Duplicados completos
duplicados_totales = df_covid.duplicated().sum()
# Duplicados en variables clave
duplicados_clave = df_covid.duplicated(subset=['DEPARTAMENTO', 'EDAD', 'SEXO', 'FECHA_RESULTADO']).sum()
print(f"Duplicados totales: {duplicados_totales}")
print(f"Duplicados en variables clave: {duplicados_clave}")

Duplicados totales: 1109602
Duplicados en variables clave: 3275748


## PASO 4: Eliminacion de registros duplicados

In [4]:
print(f"Tamaño original del DataFrame: {df_covid.shape}")
df_covid.drop_duplicates(inplace=True)
print(f"Duplicados eliminados. Nuevo tamaño del DataFrame: {df_covid.shape}")

Tamaño original del DataFrame: (4585360, 10)
Duplicados eliminados. Nuevo tamaño del DataFrame: (3475758, 10)


## PASO 5: Crear dataset limpio con conversiones de tipos
### Objetivo: Crear nuevo dataframe con tipos de datos correctos

In [5]:
df_clean_id = df_covid.copy()

# Convertir FECHA_CORTE y FECHA_RESULTADO a datetime
df_clean_id['FECHA_CORTE'] = pd.to_datetime(df_clean_id['FECHA_CORTE'], format='%Y%m%d', errors='coerce')
df_clean_id['FECHA_RESULTADO'] = pd.to_datetime(df_clean_id['FECHA_RESULTADO'], format='%Y%m%d', errors='coerce')

# Convertir EDAD a Int64
df_clean_id['EDAD'] = pd.to_numeric(df_clean_id['EDAD'], errors='coerce').astype('Int64')

# Convertir UBIGEO a string (sin .0)
df_clean_id['UBIGEO'] = pd.to_numeric(df_clean_id['UBIGEO'], errors='coerce').astype('Int64').astype(str)

# Convertir id_persona a string y renombrar a ID_PERSONA
df_clean_id['id_persona'] = pd.to_numeric(df_clean_id['id_persona'], errors='coerce').astype('Int64').astype(str)
df_clean_id.rename(columns={'id_persona': 'ID_PERSONA'}, inplace=True)

# Asignar IDs a registros sin ID_PERSONA
ids_existentes = df_clean_id['ID_PERSONA'].dropna().astype(int)
max_id = ids_existentes.max() if len(ids_existentes) > 0 else 0
filas_sin_id = df_clean_id['ID_PERSONA'].isnull().sum()
nuevos_ids = range(max_id + 1, max_id + 1 + filas_sin_id)
df_clean_id.loc[df_clean_id['ID_PERSONA'].isnull(), 'ID_PERSONA'] = [str(id) for id in nuevos_ids]
df_clean_id.sample(10)

,FECHA_CORTE,DEPARTAMENTO,PROVINCIA,DISTRITO,METODODX,EDAD,SEXO,FECHA_RESULTADO,UBIGEO,ID_PERSONA
4397980,2024-12-03,ANCASH,SANTA,CHIMBOTE,PR,47,FEMENINO,2020-11-05,21801,45342850
582662,2024-12-03,LIMA,LIMA,SANTIAGO DE SURCO,PCR,40,FEMENINO,2021-04-26,150140,42511134
2138987,2024-12-03,LIMA,LIMA,LIMA,AG,37,MASCULINO,2021-09-23,150101,43770721
2631818,2024-12-03,LIMA,HUAROCHIRI,SURCO,AG,78,FEMENINO,2022-11-30,150732,44110233
3805703,2024-12-03,CALLAO,CALLAO,LA PERLA,PCR,55,FEMENINO,2022-01-04,70104,44931068
4113756,2024-12-03,LIMA,EN INVESTIGACIÓN,EN INVESTIGACIÓN,AG,68,MASCULINO,2022-12-31,NaN,45148914
2952533,2024-12-03,JUNIN,SATIPO,PANGOA,AG,41,FEMENINO,2022-07-25,120606,44346141
4045124,2024-12-03,CUSCO,CUSCO,SANTIAGO,PCR,51,MASCULINO,2022-01-07,80106,45099221
255553,2024-12-03,LA LIBERTAD,TRUJILLO,VICTOR LARCO HERRERA,PCR,32,MASCULINO,2021-02-27,130111,42222431
3171714,2024-12-03,AREQUIPA,AREQUIPA,CERRO COLORADO,PCR,57,FEMENINO,2023-01-09,40104,44504470


In [6]:
(df_clean_id.isnull().sum() / len(df_clean_id) * 100).round(2)

FECHA_CORTE        0.00
DEPARTAMENTO       0.00
PROVINCIA          0.00
DISTRITO           0.00
METODODX           0.00
EDAD               0.01
SEXO               0.00
FECHA_RESULTADO    0.05
UBIGEO             3.87
ID_PERSONA         0.00
dtype: float64

## PASO 6: Verificación de variables clave
### Objetivo: Validar rangos y valores únicos

In [8]:
# Estadísticas de edad
df_clean_id['EDAD'].describe()

# Valores únicos en SEXO
df_clean_id['SEXO'].value_counts()

# Departamentos
df_clean_id['DEPARTAMENTO'].nunique()


print(f"Valores únicos en DEPARTAMENTO: {df_clean_id['DEPARTAMENTO'].nunique()}")
print("")
print(f"Valores únicos en {df_clean_id['SEXO'].value_counts()}")
print("")
print(f"Estadísticas de edad:\n{df_clean_id['EDAD'].describe()}")

df_clean_id['DEPARTAMENTO'].value_counts()

Valores únicos en DEPARTAMENTO: 25

Valores únicos en SEXO
FEMENINO     1794922
MASCULINO    1680836
Name: count, dtype: int64

Estadísticas de edad:
count    3475562.0
mean     41.634781
std      18.655859
min            0.0
25%           28.0
50%           40.0
75%           54.0
max          125.0
Name: EDAD, dtype: Float64


DEPARTAMENTO
LIMA             1471078
AREQUIPA          237460
PIURA             148306
LA LIBERTAD       147179
JUNIN             139996
ANCASH            135105
CUSCO             128817
CALLAO            116155
ICA               110373
LAMBAYEQUE        109127
CAJAMARCA          93173
PUNO               68084
SAN MARTIN         65410
LORETO             57657
HUANUCO            55953
TACNA              55027
AYACUCHO           51884
MOQUEGUA           50491
AMAZONAS           46493
APURIMAC           43393
UCAYALI            39486
TUMBES             30169
HUANCAVELICA       28676
PASCO              27984
MADRE DE DIOS      18282
Name: count, dtype: int64

## PASO 7: Eliminar valores nulos en variables críticas
### Objetivo: Eliminar registros con nulos en EDAD, FECHA_RESULTADO, UBIGEO


In [9]:
# Registros iniciales
print(f"Inicial: {len(df_clean_id):,}")

# Eliminar EDAD nulos
df_clean_id = df_clean_id[df_clean_id['EDAD'].notna()]

# Eliminar FECHA_RESULTADO nulos
df_clean_id = df_clean_id[df_clean_id['FECHA_RESULTADO'].notna()]

# Eliminar UBIGEO nulos (incluyendo 'nan' como string)
df_clean_id = df_clean_id[
    (df_clean_id['UBIGEO'].notna()) & 
    (df_clean_id['UBIGEO'] != 'nan') & 
    (df_clean_id['UBIGEO'] != '<NA>')
]

print(f"Final: {len(df_clean_id):,}")

Inicial: 3,475,758
Final: 3,339,541


## PASO 8: Filtrar fechas válidas (>= 2020)
### Objetivo: Eliminar registros con fechas inválidas anteriores a 2020



In [11]:
# Definir fecha de inicio de COVID-19
fecha_inicio_covid = pd.to_datetime('2020-01-01')

# Filtrar solo fechas >= 2020-01-01
df_clean_id = df_clean_id[df_clean_id['FECHA_RESULTADO'] >= fecha_inicio_covid]

## PASO 9: Validar rangos de edad (0-100 años)
### Objetivo: Eliminar edades inválidas mayores a 100 años

In [12]:
# Filtrar edades válidas (0 a 100 años)
df_clean_id = df_clean_id[(df_clean_id['EDAD'] >= 0) & (df_clean_id['EDAD'] <= 100)]

## PASO 10: Crear grupos de edad
### Objetivo: Categorizar edades en grupos epidemiológicos

In [13]:
# Definir función de categorización
def categorizar_edad(edad):
    if edad < 18:
        return '0-17 años (Menores)'
    elif edad < 30:
        return '18-29 años (Jóvenes)'
    elif edad < 45:
        return '30-44 años (Adultos Jóvenes)'
    elif edad < 60:
        return '45-59 años (Adultos)'
    elif edad < 75:
        return '60-74 años (Adultos Mayores)'
    else:
        return '75+ años (Tercera Edad)'

# Aplicar categorización
df_clean_id['GRUPO_EDAD'] = df_clean_id['EDAD'].apply(categorizar_edad)
df_clean_id['GRUPO_EDAD'].value_counts()

GRUPO_EDAD
30-44 años (Adultos Jóvenes)    1014525
45-59 años (Adultos)             770741
18-29 años (Jóvenes)             676586
60-74 años (Adultos Mayores)     433316
0-17 años (Menores)              273321
75+ años (Tercera Edad)          170251
Name: count, dtype: int64

## PASO 11: Verificación final de calidad
### Objetivo: Confirmar que el dataset está completamente limpio

#### 1. Valores Nulos: 0 nulos en todas las columnas

In [22]:
(df_clean_id.isnull().sum() / len(df_clean_id) * 100).round(2)

FECHA_CORTE        0.0
DEPARTAMENTO       0.0
PROVINCIA          0.0
DISTRITO           0.0
METODODX           0.0
EDAD               0.0
SEXO               0.0
FECHA_RESULTADO    0.0
UBIGEO             0.0
ID_PERSONA         0.0
GRUPO_EDAD         0.0
dtype: float64

#### 2. Verificacion de valores duplicados

In [23]:
# se ejecuto antes y el valor era de 13 
print(f"Registros duplicados: {df_clean_id.duplicated().sum()}")
df_clean_id.drop_duplicates(inplace=True)
print(f"Registros después de eliminar duplicados: {df_clean_id.duplicated().sum()}")

Registros duplicados: 0
Registros después de eliminar duplicados: 0


#### 3.Rangos de edad:  0-100 años (válido)

In [24]:
print(f"Edad mínima: {df_clean_id['EDAD'].min()}")
print(f"Edad máxima: {df_clean_id['EDAD'].max()}")

Edad mínima: 0
Edad máxima: 100


#### 4. Rangos de fechas:  2020-03-06 a 2024-03-04

In [25]:
#rango de fechas
print(f"Fecha mínima: {df_clean_id['FECHA_RESULTADO'].min()}")
print(f"Fecha máxima: {df_clean_id['FECHA_RESULTADO'].max()}")

Fecha mínima: 2020-03-06 00:00:00
Fecha máxima: 2024-03-04 00:00:00


#### 5. Valores categóricos:
- SEXO: 2 valores 
- DEPARTAMENTO: 25 departamentos 
- GRUPO_EDAD: 6 categorías 

In [26]:
# Valores únicos en SEXO
df_clean_id['SEXO'].value_counts()

# Departamentos
df_clean_id['DEPARTAMENTO'].nunique()


print(f"Valores únicos en DEPARTAMENTO: {df_clean_id['DEPARTAMENTO'].nunique()}")
print("")
print(f"Valores únicos en {df_clean_id['SEXO'].value_counts()}")
print("")
#MOSTRAR LAS 6 CATEGORÍAS DE EDAD
df_clean_id['GRUPO_EDAD'].value_counts().head(6)

Valores únicos en DEPARTAMENTO: 25

Valores únicos en SEXO
FEMENINO     1725974
MASCULINO    1612766
Name: count, dtype: int64



GRUPO_EDAD
30-44 años (Adultos Jóvenes)    1014525
45-59 años (Adultos)             770741
18-29 años (Jóvenes)             676586
60-74 años (Adultos Mayores)     433316
0-17 años (Menores)              273321
75+ años (Tercera Edad)          170251
Name: count, dtype: int64

#### 6. Tipos de datos: Todos correctos

In [27]:
df_clean_id.info()

<class 'pandas.DataFrame'>
Index: 3338740 entries, 0 to 4585359
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   FECHA_CORTE      datetime64[us]
 1   DEPARTAMENTO     str           
 2   PROVINCIA        str           
 3   DISTRITO         str           
 4   METODODX         str           
 5   EDAD             Int64         
 6   SEXO             str           
 7   FECHA_RESULTADO  datetime64[us]
 8   UBIGEO           str           
 9   ID_PERSONA       str           
 10  GRUPO_EDAD       str           
dtypes: Int64(1), datetime64[us](2), str(8)
memory usage: 308.9 MB


## PASO 12: Exportar Dataset Limpio

In [28]:
df_clean_id.to_csv('covid_peru_limpio.csv', index=False, sep=';', encoding='utf-8-sig')